# 00.- Librerias

In [58]:
import mysql.connector
from mysql.connector import Error
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# 01.- Cargar la Base de datos

In [59]:
def conectar_bd (name_bd:str, host_bd:str, user_bd:str, psw_bd)->tuple:
    """Intenta conectarse a una base de datos MySQL y devuelve los objetos de conexión y cursor asociados.

    Args:
        name_bd (str): Nombre de la BD a conectarse
        host_bn (ste): Nombre del host a conectarse
        user_bd (str): Nombre del usuario que se va a conectar
        psw_bd (str): Password de usuario que se va a conectar

    Returns:
        tuple: Una tupla (connection, cursor). Si la conexión falla, devuelve (None, None).
    """
    try:
        connection = mysql.connector.connect(host = host_bd,
                                        database = name_bd,
                                        user = user_bd,
                                        password = psw_bd)
        if connection.is_connected():
            db_Info = connection.get_server_info()
            print("Connected to MySQL Server version ", db_Info)
            cursor = connection.cursor()
            cursor.execute("select database();")
            record = cursor.fetchone()
            print("You're connected to database: ", record)
            
            return connection, cursor

    except Error as e:
        print("Error while connecting to MySQL", e)        
        return None, None

In [60]:
def tablas_bd (cursor)->pd.DataFrame:
    """Obtiene los nombres de todas las tablas de la base de datos activa y los devuelve en un DataFrame de pandas.

    Args:
        cursor (_type_): Objeto cursor de MySQL utilizado para ejecutar la consulta

    Returns:
        pd.DataFrame: DataFrame con una columna llamada 'tabla' que contiene
        los nombres de todas las tablas. Si ocurre un error, devuelve un
        DataFrame vacío.
    """
    try:
        sql_select_Query = "SHOW TABLES"
        cursor.execute(sql_select_Query)
        records = cursor.fetchall()
        return pd.DataFrame(records, columns=["tabla"])
    
    except Error as e:
        print("Error al obtener el nombre de todas las tablas:", e) 
        return pd.DataFrame() # Devuelve DF vacío si falla
    

In [61]:
def carga_tabla_en_df(cursor, name_tabla:str)->pd.DataFrame:
    """Carga todos los registros de una tabla de la base de datos en un DataFrame de pandas.

    Args:
        cursor (_type_): Objeto cursor de MySQL utilizado para ejecutar la consulta.
        name_tabla (str): Nombre de la tabla que se desea cargar

    Returns:
        pd.DataFrame: DataFrame con los datos de la tabla. 
        Si ocurre un error durante la ejecución de la consulta, se devuelve un DataFrame vacío.
    """
    
    try:
        sql_select_Query = f"select * from {name_tabla}"
        cursor.execute(sql_select_Query)
        records = cursor.fetchall()
        columnas = cursor.column_names        
        return pd.DataFrame(records, columns=columnas)
    
    except Error as e:
        print(f"Error al cargar la tabla {name_tabla} en DF:", e) 
        return pd.DataFrame() # Devuelve DF vacío si falla

In [62]:
def desconectar_bd(connection, cursor):
    """Cierra de forma segura el cursor y la conexión a la base de datos, siempre que ambos estén abiertos.

    Args:
        connection (_type_): Objeto de conexión MySQL
        cursor (_type_): Objeto cursor asociado a la conexión
    """
    try:
        if connection is not None and connection.is_connected():
            cursor.close()
            connection.close()
            print("MySQL connection is closed")
    except Error as e:
        print("Error while closing connection:", e)

    

In [63]:
load_dotenv()

host = os.getenv("HOST")
database = os.getenv("NAME")
user = os.getenv("USUARIO")
psw = os.getenv("PSW")

# print(f"host = {host}")
# print(f"database = {database}")
# print(f"user = {user}")
# print(f"psw = {psw}")


# 1 Conectarme a la BD
conexion, cursor = conectar_bd(database, host, user, psw)

# 2 Saber el nombre de las tablas de la BD
if cursor is None:
    print("No se pudo conectar a la BD. Revisa credenciales o permisos.")
else:
    df_tablas = tablas_bd(cursor)
    display(df_tablas)


# 3 Cargar UNA de las tablas anteriores
nombre_tabla = df_tablas.iloc[0,0]
df_pisos = carga_tabla_en_df(cursor, nombre_tabla)
display(df_pisos)


# # 4.0 Cargara todas las tablas en un diccionario
# # 4.1 Diccionario donde guardaremos todos los DataFrames 
# dic_dfs = {}

# # 4.2 Cargar todas las tabla en un DF cada una
# for nombre in df_tablas["tabla"]:
#     nom = "df_" + nombre
#     # print(nom)
#     nom_valor = carga_tabla_en_df(cursor, nombre)
#     dic_dfs[nom] = nom_valor
#     # display(nom)

# 5 desconectarme de la BD
desconectar_bd(conexion, cursor)

Connected to MySQL Server version  8.0.46
You're connected to database:  ('Equip_29',)


,tabla
0,Tourist_Accommodation


,apartment_id,name,description,host_id,neighbourhood_name,neighbourhood_district,room_type,accommodates,bathrooms,bedrooms,...,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,is_instant_bookable,reviews_per_month,country,city,insert_date
0,11964,A ROOM WITH A VIEW,Private bedroom in our attic apartment. Right ...,45553,Centro,None,Private room,2,2,1,...,100.0,100.0,100.0,100.0,100.0,FALSO,75.0,spain,malaga,31/07/2018
1,21853,Bright and airy room,We have a quiet and sunny room with a good vie...,83531,C�rmenes,Latina,Private room,1,1,1,...,90.0,100.0,100.0,80.0,90.0,FALSO,52.0,spain,madrid,10/01/2020
2,32347,Explore Cultural Sights from a Family-Friendly...,Open French doors and step onto a plant-filled...,139939,San Vicente,Casco Antiguo,Entire home/apt,4,1,2,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,142.0,spain,sevilla,29/07/2019
3,35379,Double 02 CasanovaRooms Barcelona,Room at a my apartment. Kitchen and 2 bathroom...,152232,l'Antiga Esquerra de l'Eixample,Eixample,Private room,2,2,1,...,90.0,100.0,100.0,100.0,90.0,VERDADERO,306.0,spain,barcelona,10/01/2020
4,35801,Can Torras Farmhouse Studio Suite,Lay in bed & watch sunlight change the mood of...,153805,Quart,None,Private room,5,1,2,...,100.0,100.0,100.0,100.0,100.0,FALSO,39.0,spain,girona,19/02/2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6996,27241318,ES MOLI D'EN SION - Villa with private pool in...,Enjoy the peace of the countryside in this bea...,80839530,Sa Pobla,None,Entire home/apt,10,4,5,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,7.0,spain,mallorca,23/04/2020
6997,27244243,101.108_New building apartment with two double...,Apartment in Cadaqu�s center. 1rst �floor. Ele...,151496825,Cadaqu�s,None,Entire home/apt,4,1,2,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,30/08/2018
6998,27244794,101.38_Apartment with one doble bedroom and te...,101.38.- Apartment placed Sa T�rtora � Sant An...,151496825,Cadaqu�s,None,Entire home/apt,2,1,1,...,100.0,100.0,100.0,100.0,100.0,VERDADERO,6.0,spain,girona,31/12/2019
6999,27245117,MATILLA - Fant�stico apartamento con garaje,Apartamento espacioso a 7 minutos del centro d...,137859766,Cadaqu�s,None,Entire home/apt,6,2,3,...,NaN,NaN,NaN,NaN,NaN,VERDADERO,NaN,spain,girona,31/07/2018


MySQL connection is closed


# 02.- Exportar a parquet

In [ ]:
df_pisos.to_parquet("2026_05_26_pisos_turisticos_bruto.parquet", index=True)

# 03.- Control de registros

In [64]:
# 1. TOTAL REGISTROS
total_registros = len(df_pisos)

# 2. ID ÚNICOS
id_unicos = df_pisos["apartment_id"].nunique()

# 3. ID EXTRAS (duplicados de ID menos 1)
id_extras = df_pisos["apartment_id"].duplicated().sum()

# 4. ID DUPLICADOS EXACTOS (cuenta todas las columnas iguales. Sin keep=False, cuenta SOLO las repeticiones, NO la primera aparición)
duplicados_exactos = df_pisos.duplicated(keep=False).sum()

# Mostrar resultados
print("CONTROL DE REGISTROS")
print("---------------------")
print(f"TOTAL REGISTROS: {total_registros}")
print(f"ID ÚNICOS: {id_unicos}")
print(f"ID EXTRAS: {id_extras}")
print(f"ID DUPLICADOS EXACTOS: {duplicados_exactos}")



CONTROL DE REGISTROS
---------------------
TOTAL REGISTROS: 7001
ID ÚNICOS: 6733
ID EXTRAS: 268
ID DUPLICADOS EXACTOS: 0


In [65]:
# Tipos de datos e información general
df_pisos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7001 entries, 0 to 7000
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 7001 non-null   int64  
 1   name                         6998 non-null   object 
 2   description                  6972 non-null   object 
 3   host_id                      7001 non-null   int64  
 4   neighbourhood_name           7001 non-null   object 
 5   neighbourhood_district       4241 non-null   object 
 6   room_type                    7001 non-null   object 
 7   accommodates                 7001 non-null   int64  
 8   bathrooms                    6969 non-null   object 
 9   bedrooms                     6972 non-null   object 
 10  beds                         6998 non-null   float64
 11  amenities_list               6984 non-null   object 
 12  price                        6870 non-null   float64
 13  minimum_nights    

# 04.- Duplicados. Identificar y tratar

In [66]:
# Saber cuantos apartment_id estan repetidos 1 vez, 2 veces, 3 veces, ... n veces
# Los que se repiten 1 vez son unicos 
# Los que se repiten mas de 1 vez hay que tratarlos
df_pisos["apartment_id"].value_counts().value_counts()


count
1    6472
2     254
3       7
Name: count, dtype: int64

Comprobar que para cada apartment_id repetido, tiene un insert_date diferente y si es asi, deberemos quedarnos con el registro de fecha menos antigua

In [67]:
# Ver en cada apartment_id su insert_date
df_pisos_duplicados = df_pisos[df_pisos["apartment_id"].duplicated(keep=False)][["apartment_id", "insert_date"]].sort_values("apartment_id")
df_pisos_duplicados.head(10)


,apartment_id,insert_date
22,144471,12/09/2017
23,144471,10/10/2018
24,157327,30/04/2020
25,157327,30/08/2018
50,343864,05/06/2017
51,343864,07/11/2018
89,503253,18/04/2018
90,503253,28/01/2019
224,886569,11/02/2021
225,886569,25/06/2020


In [68]:
# Comprobamos que no hay ninguna fecha de insert_date repetida en un mismos apartment_id
ids_con_fechas_repetidas = (
    df_pisos.groupby("apartment_id")["insert_date"].nunique()
    .lt(
        df_pisos.groupby("apartment_id")["insert_date"].count()
        )
)

# Obtengo los indices de los apartment_id que tienen un insert_date repetido (Trues del less than)
ids_con_fechas_repetidas = ids_con_fechas_repetidas[ids_con_fechas_repetidas].index


# IDs duplicados totales
ids_duplicados = df_pisos["apartment_id"].value_counts()
ids_duplicados = ids_duplicados[ids_duplicados > 1].index

total_ids_duplicados = len(ids_duplicados)
total_ids_con_fechas_repetidas = len(ids_con_fechas_repetidas)

print(f"Total apartment_id duplicados: {total_ids_duplicados}")
print(f"De ellos, IDs con insert_date repetido: {total_ids_con_fechas_repetidas}")


Total apartment_id duplicados: 261
De ellos, IDs con insert_date repetido: 0


In [69]:
# Comprobar el tipo de variable que es insert_date
df_pisos["insert_date"].info()


<class 'pandas.core.series.Series'>
RangeIndex: 7001 entries, 0 to 7000
Series name: insert_date
Non-Null Count  Dtype 
--------------  ----- 
7001 non-null   object
dtypes: object(1)
memory usage: 54.8+ KB


In [70]:
# Como insert_date es del tipo bbject y para quedarnos con la fecha menos antigua primero, tengo que convetirta a tipo datetime
df_pisos['insert_date'] = pd.to_datetime(df_pisos['insert_date'], format='%d/%m/%Y')

# Comprobar el tipo de variable que es insert_date
df_pisos["insert_date"].info()


<class 'pandas.core.series.Series'>
RangeIndex: 7001 entries, 0 to 7000
Series name: insert_date
Non-Null Count  Dtype         
--------------  -----         
7001 non-null   datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 54.8 KB


In [71]:
# Ordeno el df_pisos por apartment_id y por insert_date descendente para que se quede en la priemra posición el mas actual de cada uno
df_pisos_ordenado = df_pisos.sort_values(by=['apartment_id', 'insert_date'], ascending=[True, False])

# creo un df nuevo resultado de eliminar los duplicados de 'apartment_id', quedandome solo con el primero (el más actual al ordenarlo)
df_pisos_sin_duplicados = df_pisos_ordenado.drop_duplicates(subset=['apartment_id'], keep='first').copy()

# Reseteo el índice del DataFrame resultante para tenerlo estandarizado
df_pisos_sin_duplicados = df_pisos_sin_duplicados.reset_index(drop=True)

# Vuelvo a comprobar, en el df sin duplicados, cuantos apartment_id estan repetidos 1 vez, 2 veces, 3 veces, ... n veces
df_pisos_sin_duplicados["apartment_id"].value_counts().value_counts()



count
1    6733
Name: count, dtype: int64

# 05 Estandarización de formatos

In [72]:
df_pisos_sin_duplicados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6733 entries, 0 to 6732
Data columns (total 35 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   apartment_id                 6733 non-null   int64         
 1   name                         6730 non-null   object        
 2   description                  6706 non-null   object        
 3   host_id                      6733 non-null   int64         
 4   neighbourhood_name           6733 non-null   object        
 5   neighbourhood_district       4075 non-null   object        
 6   room_type                    6733 non-null   object        
 7   accommodates                 6733 non-null   int64         
 8   bathrooms                    6702 non-null   object        
 9   bedrooms                     6704 non-null   object        
 10  beds                         6730 non-null   float64       
 11  amenities_list               6717 non-null 

Transformar Variables temporales

In [73]:
# first_review_date y last_review_date de object a datatime
df_pisos_sin_duplicados['first_review_date'] = pd.to_datetime(df_pisos_sin_duplicados['first_review_date'], format='%d/%m/%Y')
df_pisos_sin_duplicados['last_review_date'] = pd.to_datetime(df_pisos_sin_duplicados['first_review_date'], format='%d/%m/%Y')

df_pisos_sin_duplicados[["first_review_date", "last_review_date"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6733 entries, 0 to 6732
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   first_review_date  5530 non-null   datetime64[ns]
 1   last_review_date   5530 non-null   datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 105.3 KB


Transformar Variables Numéricas

In [74]:
# Convierto a numerico las columnas bathrooms y bedrooms, convertiendo a NaN los errores
df_pisos_sin_duplicados['bathrooms'] = pd.to_numeric(df_pisos_sin_duplicados['bathrooms'], errors='coerce')
df_pisos_sin_duplicados['bedrooms'] = pd.to_numeric(df_pisos_sin_duplicados['bathrooms'], errors='coerce')

df_pisos_sin_duplicados[["bathrooms", "bedrooms"]].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6733 entries, 0 to 6732
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bathrooms  6702 non-null   float64
 1   bedrooms   6702 non-null   float64
dtypes: float64(2)
memory usage: 105.3 KB


Transformar variables Booleam

In [75]:
# Ver valores diferentes y frecuencia de cada uno ANTES de la transformacio
print("ANTES TRASFORMACION: ")
display(df_pisos_sin_duplicados["has_availability"].value_counts(dropna=False))

# Transformar VERDADERO a True
df_pisos_sin_duplicados['has_availability'] = df_pisos_sin_duplicados['has_availability'].replace('VERDADERO', True)

# Asumo que los valores nulls seran False por no tener disponibilidad inmediata
# df_pisos_sin_duplicados['has_availability'].fillna(False, inplace = True)
df_pisos_sin_duplicados['has_availability'] = (df_pisos_sin_duplicados['has_availability'].fillna(False))


# Ver valores diferentes y frecuencia de cada uno DESPUES de la transformacio
print("\nDESPUES TRASFORMACION: ")
display(df_pisos_sin_duplicados["has_availability"].value_counts(dropna=False))

ANTES TRASFORMACION: 


has_availability
VERDADERO    6199
None          534
Name: count, dtype: int64


DESPUES TRASFORMACION: 


C:\Users\Ignacio\AppData\Local\Temp\ipykernel_5376\2132480186.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pisos_sin_duplicados['has_availability'] = (df_pisos_sin_duplicados['has_availability'].fillna(False))


has_availability
True     6199
False     534
Name: count, dtype: int64

In [76]:
# Ver valores diferentes y frecuencia de cada uno ANTES de la transformacio
print("ANTES TRASFORMACION: ")
display(df_pisos_sin_duplicados["is_instant_bookable"].value_counts(dropna=False))

# Transformar VERDADERO a True
df_pisos_sin_duplicados['is_instant_bookable'] = df_pisos_sin_duplicados['is_instant_bookable'].replace({"VERDADERO" : True, "FALSO" : False}).infer_objects()

# # Asumo que los valores nulls seran False por no tener disponibilidad inmediata
df_pisos_sin_duplicados['is_instant_bookable'] = df_pisos_sin_duplicados['is_instant_bookable'].replace('FALSO', False)

# Ver valores diferentes y frecuencia de cada uno DESPUES de la transformacio
print("\nDESPUES TRASFORMACION: ")
display(df_pisos_sin_duplicados["is_instant_bookable"].value_counts(dropna=False))

ANTES TRASFORMACION: 


is_instant_bookable
VERDADERO    3590
FALSO        3143
Name: count, dtype: int64


DESPUES TRASFORMACION: 


C:\Users\Ignacio\AppData\Local\Temp\ipykernel_5376\3243974888.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pisos_sin_duplicados['is_instant_bookable'] = df_pisos_sin_duplicados['is_instant_bookable'].replace({"VERDADERO" : True, "FALSO" : False}).infer_objects()


is_instant_bookable
True     3590
False    3143
Name: count, dtype: int64

Variables de ratings: Tienen un intervalo maximo de 100 y 1000 cuando en la documentacion dice que es 100 para la variable "review_scores_rating" y 10 para el resto

Como no sabemos como se llega al "review_scores_rating" partiendo del resto de variables, voy a dividirla entre 10 para que coincida su maximo con el resto de variables

In [77]:
columnas = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

print("Min y MAX ANTES de normalizar:")
# display(df_pisos_sin_duplicados[columnas].agg(['min', 'max']))
display(df_pisos_sin_duplicados[columnas].describe())


# Dividir entre 10 para ponerla en la misma escala que en las otras
df_pisos_sin_duplicados['review_scores_rating'] = (df_pisos_sin_duplicados['review_scores_rating'] / 10)


print("Min y MAX DESPUES de normalizar:")
# display(df_pisos_sin_duplicados[columnas].agg(['min', 'max']))
display(df_pisos_sin_duplicados[columnas].describe())

Min y MAX ANTES de normalizar:


,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,5459.000000,5450.000000,5456.000000,5445.000000,5454.000000,5444.000000,5444.000000
mean,919.994504,94.543119,93.150660,96.277319,96.433810,95.293902,91.432770
std,85.864752,9.098797,9.759154,8.011762,7.659755,7.322268,9.347875
min,200.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
25%,890.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
50%,940.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000
75%,980.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
max,1000.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


Min y MAX DESPUES de normalizar:


,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value
count,5459.000000,5450.000000,5456.000000,5445.000000,5454.000000,5444.000000,5444.000000
mean,91.999450,94.543119,93.150660,96.277319,96.433810,95.293902,91.432770
std,8.586475,9.098797,9.759154,8.011762,7.659755,7.322268,9.347875
min,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
25%,89.000000,90.000000,90.000000,90.000000,90.000000,90.000000,90.000000
50%,94.000000,100.000000,100.000000,100.000000,100.000000,100.000000,90.000000
75%,98.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000


# 06.- Exportar df limpio a falta de Nulls

In [ ]:
df_pisos_sin_duplicados.to_parquet("20265_05_25_pisos_turisticos_sin_duplicados.parquet", index=True)

# 07.- Nulls (NaN + vacíos + espacios). Identificar

## 07.1.- Nulos

In [78]:
# Ver cuántos NULOS hay en cada columna
df_pisos_sin_duplicados.isna().sum()


apartment_id                      0
name                              3
description                      27
host_id                           0
neighbourhood_name                0
neighbourhood_district         2658
room_type                         0
accommodates                      0
bathrooms                        31
bedrooms                         31
beds                              3
amenities_list                   16
price                           121
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1203
last_review_date               1203
review_scores_rating           1274
review_scores_accuracy         1283
review_scores_cleanliness      1277
review_scores_checkin          1288
review_scores_communication 

Bathrooms: Les asignamos la mediana de baños por persona segun la capacidad

In [79]:
# Numero de bathrooms NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['bathrooms'].isna().sum()}")


# Calcula la mediana de 'bathrooms' por cada valor de 'accommodates'
mediana_bathrooms_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['bathrooms'].median()

# Asigno esa mediana a los valores NaN
def asignar_bathrooms(row):
    if pd.isnull(row['bathrooms']):
        return mediana_bathrooms_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['bathrooms'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['bathrooms']

df_pisos_sin_duplicados['bathrooms'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de bathrooms NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['bathrooms'].isna().sum()}")

'Nº de NaN ANTES = 31'

'Nº de NaN DESPUES = 0'

Bedrooms: Les asignamos la mediana de dormitorios por persona segun la capacidad

In [80]:
# Numero de bedrooms NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['bedrooms'].isna().sum()}")


# Calcula la mediana de 'bedrooms' por cada valor de 'accommodates'
mediana_bathrooms_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['bedrooms'].median()

# Asigno esa mediana a los valores NaN
def asignar_bathrooms(row):
    if pd.isnull(row['bedrooms']):
        return mediana_bathrooms_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['bedrooms'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['bedrooms']

df_pisos_sin_duplicados['bedrooms'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de bedrooms NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['bedrooms'].isna().sum()}")

'Nº de NaN ANTES = 31'

'Nº de NaN DESPUES = 0'

Beds: Les asignamos la mediana de camas por persona segun la capacidad

In [81]:
# Numero de beds NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['beds'].isna().sum()}")


# Calcula la mediana de 'beds' por cada valor de 'accommodates'
mediana_bathrooms_por_capacidad = df_pisos_sin_duplicados.groupby('accommodates')['beds'].median()

# Asigno esa mediana a los valores NaN
def asignar_bathrooms(row):
    if pd.isnull(row['beds']):
        return mediana_bathrooms_por_capacidad.get(row['accommodates'], df_pisos_sin_duplicados['beds'].median()) # Si no hay mediana para ese 'accommodates', devuelve la mediana general
    return row['beds']

df_pisos_sin_duplicados['beds'] = df_pisos_sin_duplicados.apply(asignar_bathrooms, axis=1)

# Numero de beds NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['beds'].isna().sum()}")

'Nº de NaN ANTES = 3'

'Nº de NaN DESPUES = 0'

Price: Les asignaremos la mediana del precio que tengan en la misma ciudad el mismo tipo de alojamiento

In [82]:
# Numero de beds NaN
display(f"Nº de NaN ANTES = {df_pisos_sin_duplicados['price'].isna().sum()}")

# Calculo la mediana de 'price' por 'city' y 'room_type'
mediana_price_por_city_roomtype = df_pisos_sin_duplicados.groupby(['city', 'room_type'])['price'].median()

# Función para imputar los valores nulos en 'price'
def asignar_price(row):
    if pd.isnull(row['price']):
        try:
            return mediana_price_por_city_roomtype[(row['city'], row['room_type'])]
        except KeyError:
            return df_pisos_sin_duplicados['price'].median() # Si no existe la combinación, usa la mediana general
    return row['price']

# Aplicar la imputación
df_pisos_sin_duplicados['price'] = df_pisos_sin_duplicados.apply(asignar_price, axis=1)


# Numero de beds NaN
display(f"Nº de NaN DESPUES = {df_pisos_sin_duplicados['price'].isna().sum()}")


'Nº de NaN ANTES = 121'

'Nº de NaN DESPUES = 0'

In [83]:
# Ver cuántos NULOS hay en cada columna DESPUES de la gestión
df_pisos_sin_duplicados.isna().sum()


apartment_id                      0
name                              3
description                      27
host_id                           0
neighbourhood_name                0
neighbourhood_district         2658
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                   16
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1203
last_review_date               1203
review_scores_rating           1274
review_scores_accuracy         1283
review_scores_cleanliness      1277
review_scores_checkin          1288
review_scores_communication 

## 07.2.- Cadena vacia ""

In [84]:
# Detectar cadenas vacías ""
(df_pisos == "").sum()


apartment_id                   0
name                           0
description                    0
host_id                        0
neighbourhood_name             0
neighbourhood_district         0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
beds                           0
amenities_list                 0
price                          0
minimum_nights                 0
maximum_nights                 0
has_availability               0
availability_30                0
availability_60                0
availability_90                0
availability_365               0
number_of_reviews              0
first_review_date              0
last_review_date               0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
is_instant

## 07.3.- Cadena con espacios

In [85]:
# # Detectar cadenas con espacios " " (vacíos “disfrazados”)
df_pisos_sin_duplicados.apply(
    lambda col: col.astype(str).str.strip().eq("").sum()
    if col.dtype == "object" else 0
)


apartment_id                   0
name                           0
description                    0
host_id                        0
neighbourhood_name             0
neighbourhood_district         0
room_type                      0
accommodates                   0
bathrooms                      0
bedrooms                       0
beds                           0
amenities_list                 0
price                          0
minimum_nights                 0
maximum_nights                 0
has_availability               0
availability_30                0
availability_60                0
availability_90                0
availability_365               0
number_of_reviews              0
first_review_date              0
last_review_date               0
review_scores_rating           0
review_scores_accuracy         0
review_scores_cleanliness      0
review_scores_checkin          0
review_scores_communication    0
review_scores_location         0
review_scores_value            0
is_instant

In [ ]:
# Detectar cadenas con espacios " " (vacíos “disfrazados”)
# df_pisos_sin_duplicados.apply(lambda col: col.str.strip().eq("").sum() if col.dtype == "object" else 0)


## 07.4.- Todo junto (NaN + vacíos + espacios)

In [86]:
# Detectar todo junto (NaN + vacíos + espacios)
df_pisos_sin_duplicados.apply(
    lambda col: (
        col.isna() | 
        (col.astype(str).str.strip() == "")
    ).sum()
)


apartment_id                      0
name                              3
description                      27
host_id                           0
neighbourhood_name                0
neighbourhood_district         2658
room_type                         0
accommodates                      0
bathrooms                         0
bedrooms                          0
beds                              0
amenities_list                   16
price                             0
minimum_nights                    0
maximum_nights                    0
has_availability                  0
availability_30                   0
availability_60                   0
availability_90                   0
availability_365                  0
number_of_reviews                 0
first_review_date              1203
last_review_date               1203
review_scores_rating           1274
review_scores_accuracy         1283
review_scores_cleanliness      1277
review_scores_checkin          1288
review_scores_communication 

In [87]:
# Existen NaN en algún campo de algun registro???
df_pisos_sin_duplicados.isna().any().any()


True

# 08.- Exportar df limpio final

In [89]:
df_pisos_sin_duplicados.to_parquet("2026_05_25_pisos_turisticos_limpio.parquet", index=True)